# 03 — Selected reversible variant: maximum memory-feasible batch, 50M tokens

Required experiment C. Both standard and selected-reversible batch frontiers use the **same 10-update criterion**. Search doubles until an observed failure/safety boundary, then binary-searches the passing/failing interval.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL = "https://github.com/JoeIndyGit/era-v5-session-13-reversible-llm-lab.git"
REPO_DIR = Path("/content/era-v5-session-13-reversible-llm-lab")
if not (Path.cwd() / "src").exists():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
sys.path.insert(0, str(Path.cwd()))
print("repo root:", Path.cwd())


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
from pathlib import Path
import os, sys, json
ROOT=Path.cwd()
if not (ROOT/"src").exists():
    candidates=[p.parent for p in Path("/content").glob("**/src") if p.is_dir()]
    if candidates:
        ROOT=candidates[0]; os.chdir(ROOT)
sys.path.insert(0,str(Path.cwd()))
print("repo root:",Path.cwd())

In [ ]:
from src.evidence import load_config
import json
from pathlib import Path
from src.model import ModelConfig
from src.train import find_max_stable_batch, run_experiment
cfg=load_config()
baseline_cfg=ModelConfig(**json.load(open("configs/model_baseline.json")))
sel=json.load(open("results/selected_variant.json"))
rev_cfg=ModelConfig(**json.load(open(sel["config_file"])))
print("Selected reversible variant:",sel["selected_variant"])

In [ ]:
baseline_probe=find_max_stable_batch(cfg,baseline_cfg,start_batch=cfg["batch_size"],trial_steps=10,reserve_limit=0.96)
baseline_probe


In [ ]:
rev_probe=find_max_stable_batch(cfg,rev_cfg,start_batch=cfg["batch_size"],trial_steps=10,reserve_limit=0.96)
rev_probe


In [ ]:
assert baseline_probe["search_complete"] and rev_probe["search_complete"], "Do not call a cap-limited lower bound a maximum."
max_cfg=dict(cfg); max_cfg["batch_size"]=rev_probe["largest_stable_batch"]; max_cfg["eval_batch_size"]=min(cfg["eval_batch_size"],max_cfg["batch_size"])
result=run_experiment(max_cfg,rev_cfg,"reversible_max_batch")
result

The README will report this as **maximum memory-feasible batch under the 10-update probe rule**, a narrower and more defensible claim than generic “stable batch.”